<div align="center">

# Hoja de Trabajo 2

**Sofia Garcia 22210**  
**Julio Garcia Salas 22076**

</div>

---

## Ejercicio 1

### 1) Escalabilidad en ABM cuando los atributos **sí** están interconectados (vs. heterogeneidad ortogonal)

Cuando los atributos de los agentes **no** se afectan entre sí (heterogeneidad “ortogonal”), la simulación se vuelve más fácil de escalar: se puede dividir la población en partes, calcular riesgos por separado y casi no se necesita coordinar entre hilos o máquinas. En cambio, si los atributos **sí** están fuertemente conectados (por ejemplo, *ingresos* que cambian mucho el riesgo **según** la *edad*), cada cambio en un atributo toca a los demás y obliga a recalcular más cosas.

En términos sencillos, con ortogonalidad el riesgo suele descomponerse en piezas independientes:

$$
h(t\mid \mathbf{x}) \;\approx\; h_0(t)\,\prod_j \phi_j(x_j)
$$

Con interdependencia fuerte, aparece un **término de interacción** (lo que hace que el efecto de un atributo dependa del otro), así que el riesgo ya no es “por partes”:

$$
h(t\mid \mathbf{x}) \;\propto\; \exp\!\big(\beta_1 x_1 + \beta_2 x_2 + \beta_{12}\,x_1x_2\big)
$$

**Qué cambia en la práctica (y por qué se complica escalar):**
- **Más recomputación:** si cambia ingresos, se altera el riesgo “con edad” y hay que recalcular; ya no se puede reusar tanto lo que estaba cacheado.
- **Más coordinación:** las particiones dejan de ser tan independientes; se necesita comunicar cambios entre bloques (pierde paralelismo “barato”).
- **Más memoria:** se guardan relaciones conjuntas (no solo promedios por atributo), lo que crece más rápido.
- **Calibración más pesada:** no basta con ajustar un promedio por atributo; se ajustan **interacciones** (lo que toma más iteraciones).

**Ideas simples para no perder toda la escalabilidad:**
- Preparar una **población sintética correlacionada** al inicio (ya con las dependencias) y luego **actualizar en lotes** cada $\Delta t$ en vez de cada microcambio.
- **Particionar por comunidades** (grupos con mucha interacción dentro y poca fuera) para reducir la comunicación entre particiones.
- Usar **caché con invalidación selectiva** (solo se invalida lo afectado, no todo el perfil del agente).
- Aproximar riesgos conjuntos con **modelos ligeros** (por ejemplo, una regresión corta) para evitar recomputos costosos en cada paso.

En resumen, cuando hay interdependencia fuerte, se pierde parte del “turbo” de la paralelización fácil. Se gana realismo, pero se paga con más cálculo, más coordinación y más memoria. Aun así, con las estrategias anteriores se conserva buena parte del rendimiento sin sacrificar la idea central del modelo.

---

### 2) Duraciones **fijas** (tiempos de espera) vs. **frecuencias** (riesgo sin memoria) en crónicas: ¿qué se gana y qué se pierde?

Aquí se comparan dos formas de programar “cuándo cambia de estado” un agente:

- **Frecuencias (sin memoria):** el tiempo hasta el evento es exponencial y el “peligro” por unidad de tiempo es constante. Es la versión más simple:
  
  $$
  T \sim \mathrm{Exp}(\lambda),\quad h(t)=\lambda,\quad S(t)=e^{-\lambda t}
  $$

  Ventajas: muy fácil de simular y de calibrar con tasas agregadas. Desventaja: **no recuerda** cuánto tiempo lleva el agente en el estado, lo que puede ser poco realista en crónicas donde el riesgo suele **crecer** con el tiempo.

- **Duraciones fijas (o casi fijas):** se mantiene al agente en un estado por un tiempo típico y luego se cambia. Para evitar que todos cambien exactamente al mismo tiempo, se usa una distribución con poca variabilidad (por ejemplo, Gamma/Erlang):

  $$
  T \sim \mathrm{Gamma}(k,\theta),\quad \mathbb{E}[T]=k\theta,\quad \mathrm{CV}\approx \frac{1}{\sqrt{k}}
  $$

  Ventajas: refleja mejor “fases mínimas” o “latencias” y evita que algunos salten **demasiado** pronto. Desventajas: se debe llevar un pequeño “reloj” por agente y cuidar que no haya sincronías perfectas (se soluciona con un poco de *jitter* o usando Erlang).

**Cuándo conviene cada una (pensando en crónicas):**
- Si se sospecha que el riesgo **aumenta** con el tiempo en estado (muy común en crónicas), conviene una distribución **con memoria**. Una opción práctica es **Weibull** con parámetro de forma $k>1$ (hazard creciente):

  $$
  h(t;k,\lambda) \;=\; k\,\lambda^{k}\,t^{k-1}\quad (k>1 \Rightarrow h(t)\ \text{creciente})
  $$

- Si se está en fase de **prototipo** o solo se tienen datos muy agregados, la exponencial es un buen punto de partida por su simplicidad. Más adelante se puede pasar a Gamma/Weibull para ganar realismo.

- **Duraciones (fijas/casi fijas):** más realismo temporal en crónicas y control de variabilidad; requieren llevar un “reloj” por agente y añadir un poco de aleatoriedad para no sincronizar picos.
- **Frecuencias (exponencial):** muy simple y rápida; útil para comenzar, pero puede **distorsionar** si el riesgo depende de cuánto tiempo se ha permanecido en el estado.

---  

<div align="center">

# Hoja de Trabajo 2

**Sofia Garcia 22210**  
**Julio Garcia Salas 22076**

</div>

---

## Ejercicio 2

> **Nota de contexto (propio del trabajo):** cuando aplica, se fundamenta con lo ya construido en los simuladores:  
> - **HospitalSimDiscrete** (pasos diarios) y **HospitalSimEvent** (cola de eventos) guardan historia por agente (`infector_id`, `secondary_infections`, tiempos `t_E_to_I`, `t_I_to_R`, `ward`, `role`, etc.).  
> - El diseño de **Vacunación** lleva log por agente (`ts_eligible`, `vaccinated`, `dose_num`, `reminders_sent`, `access_level`, distancia a clínica, no-show, etc.) y genera métricas de equidad (curva de **Lorenz** y **Gini**), además de **tiempos de espera** por subpoblación.

---

### 1) Evidencia empírica de que el **historial a nivel de agente** ofrece mejores insumos de política que solo recuentos agregados

En modelos por compartimentos se ven totales (S, I, R…), mientras que en un ABM como los usados aquí **se guarda la trayectoria individual**: quién infectó a quién (`infector_id`), cuántos secundarios generó (`secondary_infections`), en qué **ward** ocurrió, si la persona es **staff** o **patient**, y si se movió de sala. Con eso se pueden evaluar políticas **que dependen de trayectorias y redes**, no solo de promedios.

- **Trazabilidad práctica para políticas TTQ (Test–Trace–Quarantine):** al tener enlaces de contagio, se calcula un “verdadero” por caso (ya lo hace `true_Reff`) y se identifican focos (“wards” o roles). En aplicaciones reales, un ABM calibrado a **Seattle/King County** mostró que con niveles altos pero alcanzables de testeo y rastreo era posible **contener** aun con movilidad laboral/comunitaria restablecida; esto se apoyó en historia por agente (quién contacta a quién, demoras, cumplimiento), algo que el agregado puro no diferencia.  [Nature](https://www.nature.com/articles/s41467-021-23276-9?) [PMC](https://pmc.ncbi.nlm.nih.gov/articles/PMC8341708/)

- **Brotes validados con datos reales:** se han reproducido brotes de **sarampión** (Schull, Irlanda, 2012) con ABM dirigidos por datos abiertos, comparando con lo observado y usando la estructura por agente para entender **qué combinaciones** (edad, cobertura, estructura del lugar) explican la dinámica. La validación directa contra brote real respalda que la granularidad por agente ofrece diagnósticos de política más finos que mirar solo series agregadas.  [PMC](https://pmc.ncbi.nlm.nih.gov/articles/PMC6300276/)

- **Marco de validación de ABM:** existen metodologías para probar/validar ABM (número de corridas, sensibilidad, comparación con datos); se documenta su uso para evaluar intervenciones y “timing” en pandemias, con discusión explícita del costo computacional y de cuándo merece la pena la granularidad.  [arrow.tudublin.ie](https://arrow.tudublin.ie/cgi/viewcontent.cgi?article=1305&context=scschcomcon&utm_source=chatgpt.com) [MDPI](https://www.mdpi.com/1999-4893/15/8/270)

**Cómo se aterriza a este trabajo:**  
- En **HospitalSimEvent** ya se registra el **infector** y los **secundarios** de cada agente → se puede medir la distribución por **rol** y **ward**, priorizando **rastreo** donde más rinde (algo no visible con SIR agregados).  
- En la **simulación de vacunas**, el log por agente permite medir **no-show**, **esperas** y **cobertura por acceso**; además, se graficó una **curva de Lorenz** y se calculó **Gini** para ver **equidad** (esto es insumo de política, no un simple total).  
- Con esos históricos se pueden probar reglas como “si `access_level`=3 y distancia baja, subir prioridad”, y verificar impacto **real** en espera y cobertura, no solo en la curva total de dosis.

---

### 2) Cómo el **modelado compositivo** (p. ej., gráficos de estados jerárquicos) enfrenta la **maldición de la dimensionalidad** para **comorbilidades**, vs. stock–flow tradicional

El problema clásico: si en un modelo stock–flow se estratifica por **edad**, **ingresos**, **EPOC**, **diabetes**, etc., el número de compartimentos **explota**. Si cada rasgo \(i\) tiene \(m_i\) categorías y el modelo base tiene \(N_{\text{base}}\) compartimentos:

$$
N_{\text{comp}} \;\approx\; N_{\text{base}} \times \prod_{i=1}^{K} m_i,
$$

lo que complica el código, los datos requeridos y la **calibración**. Se han propuesto tres líneas complementarias para contener esa explosión:

1) **Descomposición en variables locales (CTBNs):** los **Continuous-Time Bayesian Networks** modelan cada rasgo como un **proceso local** en tiempo continuo con dependencias dirigidas. En vez de una mega-tabla de estados para todas las combinaciones, se especifican **transiciones locales** y sus dependencias, factorando la dinámica global. Esto reduce la enumeración explícita de combinaciones y ataca la explosión dimensional desde la representación.  [arXiv](https://arxiv.org/abs/1301.0591) [Computer Science and Engineering](https://www.cs.ucr.edu/~cshelton/papers/docs/ctbn.pdf)

2) **Modelado por reglas (rule-based):** se describen **patrones** (“si comorbilidad A y edad alta ⇒ subir riesgo de hospitalización”) sin crear un compartimento distinto por cada cruce. Este enfoque se creó precisamente para combatir la **combinatoria** en biología de sistemas y ha madurado en plataformas como **Kappa**.  [PMC](https://pmc.ncbi.nlm.nih.gov/articles/PMC3947470/) [Oxford Academic](https://academic.oup.com/bioinformatics/article/34/13/i583/5045802) [PLOS](https://journals.plos.org/plosone/article?id=10.1371%2Fjournal.pone.0114296&)

3) **Composición/estratificación modular en stock–flow:** marcos recientes (StockFlow.jl, categóricos) permiten **componer** diagramas y **estratificar** sin reescribir todo: se ensamblan piezas (edad, comorbilidad) como módulos, manteniendo la claridad y reusabilidad, y evitando duplicación masiva al añadir nuevas dimensiones.  [arXiv](https://arxiv.org/abs/2205.08373) [math.ucr.edu](https://math.ucr.edu/home/baez/compositional_modeling/ACT2023_web.pdf)

Además, por el lado numérico, se ha avanzado en **métodos tensoriales** (producto de Kronecker, **tensor-train**) que explotan estructura de baja-rango para resolver modelos de gran dimensión sin enumerar todo el espacio de estados. Aunque más técnicos, ayudan a que la computación sea viable cuando se combinan muchas dimensiones.  [SIAM Ebooks](https://epubs.siam.org/doi/10.1137/090752286) [ScienceDirect](https://www.sciencedirect.com/science/article/pii/S0096300323004599)

**Cómo se aterriza a este trabajo:**  
- En los ABM ya implementados **no** se creó un compartimento por cada combinación (edad × rol × acceso × riesgo × reticencia × ward…). Se representó cada rasgo como **campo del agente** y las reglas/eventos leen esos campos cuando toca. Eso es, en espíritu, **compositivo**: se “enchufan” rasgos sin multiplicar compartimentos.  
- En **Vacunación**, la prioridad se arma con una **llave** que combina **tier (edad/rol)**, **riesgo**, **acceso**, **distancia** y **tiempo en espera**. Se compuso una política a partir de piezas, sin construir una tabla de 1000 casillas.  
- Si mañana se añade una comorbilidad (p. ej., **diabetes**), en el ABM **solo** se agrega el campo y la regla que ajusta riesgo; en stock–flow estratificado habría que multiplicar casillas.  
- Si se quisiera llevar estas ideas a un stock–flow formal, los trabajos sobre **composición/estratificación** ofrecen una ruta para modularizar sin reescribir todo el diagrama al agregar comorbilidades o dimensiones socioeconómicas.  [arXiv](https://arxiv.org/abs/2205.08373)

**Cierre:** el modelado compositivo (variables locales, reglas, y/o diagramas componibles) **reduce** la necesidad de enumerar todas las combinaciones de comorbilidades, enfrentando la maldición de la dimensionalidad mejor que el stock–flow estratificado tradicional; esto encaja con la forma en que ya se modeló en este trabajo (rasgos como campos del agente y reglas/eventos que los usan).  [arXiv](https://arxiv.org/abs/1301.0591) [PMC](https://pmc.ncbi.nlm.nih.gov/articles/PMC3947470/)

---